In [1]:
import cv2 as cv
import numpy as np
import os

In [2]:
def extrage_careu(image):
    image = cv.cvtColor(image,cv.COLOR_BGR2HSV)
    low = (14, 0, 0)
    high = (255, 255, 255)
    mask_hsv = cv.inRange(image, low, high)
    contours, _ = cv.findContours(mask_hsv,  cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    max_area = 0
   
    for i in range(len(contours)):
        if(len(contours[i]) >3):
            possible_top_left = None
            possible_bottom_right = None
            for point in contours[i].squeeze():
                if possible_top_left is None or point[0] + point[1] < possible_top_left[0] + possible_top_left[1]:
                    possible_top_left = point

                if possible_bottom_right is None or point[0] + point[1] > possible_bottom_right[0] + possible_bottom_right[1] :
                    possible_bottom_right = point

            diff = np.diff(contours[i].squeeze(), axis = 1)
            possible_top_right = contours[i].squeeze()[np.argmin(diff)]
            possible_bottom_left = contours[i].squeeze()[np.argmax(diff)]
            if cv.contourArea(np.array([[possible_top_left],[possible_top_right],[possible_bottom_right],[possible_bottom_left]])) > max_area:
                max_area = cv.contourArea(np.array([[possible_top_left],[possible_top_right],[possible_bottom_right],[possible_bottom_left]]))
                top_left = possible_top_left
                bottom_right = possible_bottom_right
                top_right = possible_top_right
                bottom_left = possible_bottom_left

    width = 1960
    height = 1960
    
    image_copy = cv.cvtColor(image.copy(),cv.COLOR_HSV2BGR)
    cv.circle(image_copy,tuple(top_left),20,(0,0,255),-1)
    cv.circle(image_copy,tuple(top_right),20,(0,0,255),-1)
    cv.circle(image_copy,tuple(bottom_left),20,(0,0,255),-1)
    cv.circle(image_copy,tuple(bottom_right),20,(0,0,255),-1)

    puzzle = np.array([top_left,top_right,bottom_right,bottom_left], dtype = "float32")
    destination_of_puzzle = np.array([[0,0],[width,0],[width,height],[0,height]], dtype = "float32")

    M = cv.getPerspectiveTransform(puzzle,destination_of_puzzle)

    result = cv.warpPerspective(image, M, (width, height))
    result = cv.cvtColor(result,cv.COLOR_HSV2BGR)

    crop_margin = 255
    cropped_result = result[crop_margin:height-crop_margin, crop_margin:width-crop_margin]

    final_result = cv.resize(cropped_result, (width, height), interpolation=cv.INTER_LINEAR)
    
    return final_result

In [3]:
lines_horizontal=[]
for i in range(0,1961,140):
    l=[]
    l.append((0,i))
    l.append((1959,i))
    lines_horizontal.append(l)

In [4]:
lines_vertical=[]
for i in range(0,1961,140):
    l=[]
    l.append((i,0))
    l.append((i,1959))
    lines_vertical.append(l)

In [5]:
def determina_configuratie_careu_ox(img1, img2, lines_horizontal,lines_vertical):
    differences = np.zeros((14, 14), dtype='float')
    for i in range(len(lines_horizontal)-1):
        for j in range(len(lines_vertical)-1):
            y_min = lines_vertical[j][0][0] + 15
            y_max = lines_vertical[j + 1][1][0] - 15
            x_min = lines_horizontal[i][0][1] + 15
            x_max = lines_horizontal[i + 1][1][1] - 15
            patch1 = img1[x_min:x_max, y_min:y_max].copy()
            patch2 = img2[x_min:x_max, y_min:y_max].copy()
            diff = cv.absdiff(patch1, patch2)
            differences[i, j] = np.sum(diff)
    
    return differences

In [6]:
def identifică_patratel_modificat(differences):
    idx = np.unravel_index(np.argmax(differences), differences.shape)
    return idx

In [7]:
def clasifica_cifra(patch):
    maxi = -np.inf
    cifra_detectata = -1
    
    for j in range(10):
        img_template = cv.imread(f'imagini/{j}.jpg', cv.IMREAD_GRAYSCALE)
        
        template_height, template_width = img_template.shape
        patch_resized = cv.resize(patch, (template_width, template_height), interpolation=cv.INTER_AREA)

        corr = cv.matchTemplate(patch_resized, img_template, cv.TM_CCOEFF_NORMED)
        corr = np.max(corr)
        
        if corr > maxi:
            maxi = corr
            cifra_detectata = j
    
    return cifra_detectata

In [8]:
def clasifica_numar(patch):
    patch_gray = cv.cvtColor(patch, cv.COLOR_BGR2GRAY)
    _, thresh = cv.threshold(patch_gray, 127, 255, cv.THRESH_BINARY_INV)
    contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=lambda c: cv.boundingRect(c)[0])
    numar = []
    
    for contour in contours:
        x, y, w, h = cv.boundingRect(contour)
        if w < 10 or h < 10:
            continue
        cifra_patch = patch_gray[y:y+h, x:x+w]
        cifra_detectata = clasifica_cifra(cifra_patch)
        numar.append(str(cifra_detectata))
 
    return int("".join(numar))

In [9]:
valori = [[None for _ in range(14)] for _ in range(14)]
valori[6][6] = 1
valori[6][7] = 2
valori[7][6] = 3
valori[7][7] = 4

bonus = [[1 for _ in range(14)] for _ in range(14)]
bonus[0][0] = 3
bonus[0][6] = 3
bonus[0][7] = 3
bonus[0][13] = 3
bonus[13][0] = 3
bonus[13][6] = 3
bonus[13][7] = 3
bonus[13][13] = 3
bonus[6][0] = 3
bonus[6][13] = 3
bonus[7][0] = 3
bonus[7][13] = 3

bonus[1][1] = 2
bonus[12][12] = 2
bonus[2][2] = 2
bonus[11][11] = 2
bonus[3][3] = 2
bonus[10][10] = 2
bonus[4][4] = 2
bonus[9][9] = 2
bonus[9][4] = 2
bonus[10][3] = 2
bonus[11][2] = 2
bonus[12][1] = 2
bonus[4][9] = 2
bonus[3][10] = 2
bonus[2][11] = 2
bonus[1][12] = 2

constrangeri = [[None for _ in range(14)] for _ in range(14)]
constrangeri[1][4] = '%'
constrangeri[12][4] = '%'
constrangeri[1][9] = '%'
constrangeri[12][9] = '%'
constrangeri[4][1] = '%'
constrangeri[4][12] = '%'
constrangeri[9][1] = '%'
constrangeri[9][12] = '%'
constrangeri[2][5] = '-'
constrangeri[2][8] = '-'
constrangeri[11][5] = '-'
constrangeri[11][8] = '-'
constrangeri[5][2] = '-'
constrangeri[5][11] = '-'
constrangeri[8][2] = '-'
constrangeri[8][11] = '-'
constrangeri[3][6] = '+'
constrangeri[4][7] = '+'
constrangeri[6][4] = '+'
constrangeri[6][10] = '+'
constrangeri[7][3] = '+'
constrangeri[7][9] = '+'
constrangeri[9][6] = '+'
constrangeri[10][7] = '+'
constrangeri[3][7] = '*'
constrangeri[4][6] = '*'
constrangeri[6][3] = '*'
constrangeri[6][9] = '*'
constrangeri[7][4] = '*'
constrangeri[7][10] = '*'
constrangeri[9][7] = '*'
constrangeri[10][6] = '*'

In [10]:
def scor_runda(linie, coloana):
    ecuatii_multiple = 0

    if linie - 2 >= 0 and valori[linie - 1][coloana] != None and valori[linie - 2][coloana] != None: #verific ecuatia de deasupra
        satisfacut = False
        if constrangeri[linie][coloana] == '%' or constrangeri[linie][coloana] == None:
            if (valori[linie - 1][coloana] != 0 and valori[linie - 2][coloana] // valori[linie - 1][coloana] == valori[linie][coloana]) or (valori[linie - 2][coloana] != 0 and valori[linie - 1][coloana] // valori[linie - 2][coloana] == valori[linie][coloana]):
                satisfacut = True
        if constrangeri[linie][coloana] == '-' or constrangeri[linie][coloana] == None:
            if abs(valori[linie - 2][coloana] - valori[linie - 1][coloana]) == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '+' or constrangeri[linie][coloana] == None:
            if valori[linie - 2][coloana] + valori[linie - 1][coloana] == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '*' or constrangeri[linie][coloana] == None:
            if valori[linie - 2][coloana] * valori[linie - 1][coloana] == valori[linie][coloana]:
                satisfacut = True
        if satisfacut == True:
            ecuatii_multiple += 1

    if coloana + 2 <= 13 and valori[linie][coloana + 1] != None and valori[linie][coloana + 2] != None: #verific ecuatia din dreapta
        satisfacut = False
        if constrangeri[linie][coloana] == '%' or constrangeri[linie][coloana] == None:
            if (valori[linie][coloana + 2] != 0 and valori[linie][coloana + 1] // valori[linie][coloana + 2] == valori[linie][coloana]) or (valori[linie][coloana + 1] != 0 and valori[linie][coloana + 2] // valori[linie][coloana + 1] == valori[linie][coloana]):
                satisfacut = True
        if constrangeri[linie][coloana] == '-' or constrangeri[linie][coloana] == None:
            if abs(valori[linie][coloana + 1] - valori[linie][coloana + 2]) == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '+' or constrangeri[linie][coloana] == None:
            if valori[linie][coloana + 1] + valori[linie][coloana + 2] == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '*' or constrangeri[linie][coloana] == None:
            if valori[linie][coloana + 1] * valori[linie][coloana + 2] == valori[linie][coloana]:
                satisfacut = True
        if satisfacut == True:
            ecuatii_multiple += 1

    if coloana - 2 >= 0 and valori[linie][coloana - 1] != None and valori[linie][coloana - 2] != None: #verific ecuatia din stanga
        satisfacut = False
        if constrangeri[linie][coloana] == '%' or constrangeri[linie][coloana] == None:
            if (valori[linie][coloana - 2] != 0 and valori[linie][coloana - 1] // valori[linie][coloana - 2] == valori[linie][coloana]) or (valori[linie][coloana - 1] != 0 and valori[linie][coloana - 2] // valori[linie][coloana - 1] == valori[linie][coloana]):
                satisfacut = True
        if constrangeri[linie][coloana] == '-' or constrangeri[linie][coloana] == None:
            if abs(valori[linie][coloana - 1] - valori[linie][coloana - 2]) == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '+' or constrangeri[linie][coloana] == None:
            if valori[linie][coloana - 1] + valori[linie][coloana - 2] == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '*' or constrangeri[linie][coloana] == None:
            if valori[linie][coloana - 1] * valori[linie][coloana - 2] == valori[linie][coloana]:
                satisfacut = True
        if satisfacut == True:
            ecuatii_multiple += 1

    if linie + 2 <= 13 and valori[linie + 1][coloana] != None and valori[linie + 2][coloana] != None: #verific ecuatia de dedesupt
        satisfacut = False
        if constrangeri[linie][coloana] == '%' or constrangeri[linie][coloana] == None:
            if (valori[linie + 1][coloana] != 0 and valori[linie + 2][coloana] // valori[linie + 1][coloana] == valori[linie][coloana]) or (valori[linie + 2][coloana] != 0 and valori[linie + 1][coloana] // valori[linie + 2][coloana] == valori[linie][coloana]):
                satisfacut = True
        if constrangeri[linie][coloana] == '-' or constrangeri[linie][coloana] == None:
            if abs(valori[linie + 2][coloana] - valori[linie + 1][coloana]) == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '+' or constrangeri[linie][coloana] == None:
            if valori[linie + 2][coloana] + valori[linie + 1][coloana] == valori[linie][coloana]:
                satisfacut = True
        if constrangeri[linie][coloana] == '*' or constrangeri[linie][coloana] == None:
            if valori[linie + 2][coloana] * valori[linie + 1][coloana] == valori[linie][coloana]:
                satisfacut = True
        if satisfacut == True:
            ecuatii_multiple += 1

    return valori[linie][coloana] * ecuatii_multiple * bonus[linie][coloana]

In [11]:
# de modificat
PATH = 'evaluare\\fake_test\\'

numar_la_litera = {
    1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'E', 6: 'F', 7: 'G',
    8: 'H', 9: 'I', 10: 'J', 11: 'K', 12: 'L', 13: 'M', 14: 'N'
}

files_to_clear = [
    "343_Capitanu_Andreea/1_scores.txt",
    "343_Capitanu_Andreea/2_scores.txt",
    "343_Capitanu_Andreea/3_scores.txt",
    "343_Capitanu_Andreea/4_scores.txt",
]

for file_path in files_to_clear:
    if os.path.exists(file_path):
        with open(file_path, "w") as f:
            pass

img1 = cv.imread('imagini/' + "board.jpg")
result1 = extrage_careu(img1)


lista_idx = []
current_match = None
os.makedirs('343_Capitanu_Andreea', exist_ok=True)

player_scores = 0
last_round_limit = 1
current_player_name = None
next_player_name = None
current_round_limit = None
turns_iterator = None
files = sorted(os.listdir(PATH))

match_score_files = {}

for file in files:
    if file[-3:] == 'jpg':
        match_number = int(file.split('_')[0])
        rnd = file.split('_')[1].split('.')[0]
        round_number = int(rnd)

        if current_match is None or match_number != current_match:
        
            if current_match is not None and match_number != current_match:
                with open(output_file1, "a") as output_task3:
                    output_task3.write(f"{current_player_name} {last_round_limit} {player_scores}\n")
                    
            valori = [[None for _ in range(14)] for _ in range(14)]
            valori[6][6] = 1
            valori[6][7] = 2
            valori[7][6] = 3
            valori[7][7] = 4    

            current_match = match_number
            
            output_file1 = f"343_Capitanu_Andreea/{match_number}_scores.txt"

            turns_file = f"{PATH}{match_number}_turns.txt"
            if os.path.exists(turns_file):
                with open(turns_file, "r") as tf:
                    turns_iterator = iter(tf.readlines())

            player_scores = 0
            current_round_limit = None

            if turns_iterator:
                line = next(turns_iterator, None)
                if line:
                    current_player_name, _ = line.strip().split()

                line = next(turns_iterator, None)
                if line:
                    next_player_name, round_limit = line.strip().split()
                    last_round_limit = 1
                    current_round_limit = int(round_limit)

            img1 = cv.imread('imagini/' + "board.jpg")
            result1 = extrage_careu(img1)

        img2 = cv.imread(PATH + file)
        result2 = extrage_careu(img2)

        idx = identifică_patratel_modificat(determina_configuratie_careu_ox(result1, result2, lines_horizontal, lines_vertical))

        y_min = lines_vertical[idx[1]][0][0] + 10
        y_max = lines_vertical[idx[1] + 1][1][0] - 10
        x_min = lines_horizontal[idx[0]][0][1] + 10
        x_max = lines_horizontal[idx[0] + 1][1][1] - 10

        patch = result2[x_min:x_max, y_min:y_max].copy()
        numar_detectat = clasifica_numar(patch)

        idx = (int(idx[0]), int(idx[1]))
        valori[idx[0]][idx[1]] = numar_detectat

        if current_round_limit is not None and round_number < current_round_limit:
            player_scores += scor_runda(idx[0], idx[1])
            
        if current_round_limit is not None and round_number == current_round_limit:
            with open(output_file1, "a") as output_task3:
                output_task3.write(f"{current_player_name} {last_round_limit} {player_scores}\n")

            line = next(turns_iterator, None)
            if line:
                current_player_name = next_player_name
                next_player_name, round_limit = line.strip().split()
                last_round_limit = current_round_limit
                current_round_limit = int(round_limit)
                player_scores = scor_runda(idx[0], idx[1])

            else:
                last_round_limit = current_round_limit
                current_player_name = next_player_name
                current_round_limit = 51
                player_scores = scor_runda(idx[0], idx[1])
            
        idx = (int(idx[0]) + 1, numar_la_litera[int(idx[1]) + 1])
        output_file = f"343_Capitanu_Andreea/{match_number}_{rnd}.txt"
        with open(output_file, "w") as output_task:
            output_task.write(f"{idx[0]}{idx[1]} {numar_detectat}\n")

        result1 = result2

with open(output_file1, "a") as output_task3:
    output_task3.write(f"{current_player_name} {last_round_limit} {player_scores}\n")
